# Prompt Engineering Lab — TechFlow Solutions

Diagnose and fix three unreliable chatbot prompts (sentiment analysis, product descriptions, data extraction)
by **running each prompt many times**, measuring consistency, and iterating v1 → v2 → v3.

**How consistency is measured** (the same definition is used for every version, so numbers are comparable):

| Task | Consistency | Also reported |
|---|---|---|
| Sentiment | share of runs giving the *identical* normalised answer (modal share) | `format_valid` = exactly one of positive/negative/neutral; `correct` = right label |
| Product description | share of runs that pass the **same 5-check spec** (price present, exactly 3 bullets, 40–100 words, no markdown, no emoji/exclamation marks). Free text can never be byte-identical, so format conformance is the meaningful measure. | `unique` = distinct raw outputs |
| Data extraction | modal share of the canonical parsed JSON (invalid output falls back to its raw text, so it essentially never matches) | `format_valid` = `json.loads` works on the raw output with the exact schema keys; `correct` = values match ground truth |

All v1/v2/v3 runs use the API default **temperature = 1.0** so improvements come from the prompt, not from turning randomness off
(temperature is isolated in Bonus Challenge 1). Model: `gpt-4o-mini`.

## Part 1 — Starting simple
### Step 1: Environment & helper functions

In [1]:
import os, re, json, time, hashlib
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing - put it in .env"

client = OpenAI(max_retries=5)
MODEL = "gpt-4o-mini"
DEFAULT_TEMPERATURE = 1.0

# Every batch of API calls is cached on disk so re-running the notebook reproduces the exact same numbers.
# Set USE_CACHE = False (or delete results/cache) to re-sample live from the API.
USE_CACHE = True
CACHE_DIR = Path("results/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120, "display.width", 200)


def call_openai(prompt, temperature=DEFAULT_TEMPERATURE, max_tokens=600):
    """Single chat-completion call; returns the response text."""
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()


def run_many(prompt, n, label, temperature=DEFAULT_TEMPERATURE, max_tokens=600, max_workers=8):
    """Run the same prompt n times concurrently. Results are cached per (label, n, prompt, temperature)."""
    key = hashlib.sha256(json.dumps([MODEL, prompt, temperature, max_tokens, n]).encode()).hexdigest()[:12]
    path = CACHE_DIR / f"{label}__n{n}__{key}.json"
    if USE_CACHE and path.exists():
        return json.loads(path.read_text())
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        responses = list(ex.map(lambda _: call_openai(prompt, temperature, max_tokens), range(n)))
    path.write_text(json.dumps(responses, indent=1))
    return responses


print(f"Setup OK - model={MODEL}, temperature={DEFAULT_TEMPERATURE}, cache={'on' if USE_CACHE else 'off'}")
print("Smoke test:", call_openai("Reply with the single word: ready", temperature=0, max_tokens=5))

Setup OK - model=gpt-4o-mini, temperature=1.0, cache=on


Smoke test: ready


### Evaluation helpers
One evaluator per task. Each returns, per response: `key` (what we compare for identical-ness), `valid` (format ok) and `correct` (content ok).
`run_experiment` runs N samples, scores them, and appends a row to the global results table.

In [2]:
RESULTS = []          # one row per experiment -> results/summary.csv
RAW = {}              # (task, version, case, n) -> list of responses, for failure-pattern inspection


def normalize(text):
    return re.sub(r"[^a-z0-9 ]", "", text.lower()).strip()


# ---------- Sentiment ----------
LABELS = {"positive", "negative", "neutral"}

def sentiment_eval(expected):
    def ev(resp):
        key = normalize(resp)
        found = LABELS & set(re.findall(r"[a-z]+", resp.lower()))
        return dict(key=key, valid=key in LABELS, correct=(found == {expected}))
    return ev


# ---------- Product description ----------
EMOJI = re.compile("[☀-➿\U0001F000-\U0001FFFF]")

def product_checks(resp, price):
    bullets = [l for l in resp.splitlines() if re.match(r"\s*([-*•])\s+\S", l)]
    words = len(resp.split())
    return {
        "price_present": price in resp,
        "exactly_3_bullets": len(bullets) == 3,
        "length_40_100_words": 40 <= words <= 100,
        "no_markdown": "**" not in resp and not re.search(r"^\s*#", resp, re.M),
        "no_emoji_or_exclaim": not EMOJI.search(resp) and "!" not in resp,
    }

def product_eval(price):
    def ev(resp):
        checks = product_checks(resp, price)
        ok = all(checks.values())
        failed = [k for k, v in checks.items() if not v]
        return dict(key="PASS" if ok else "FAIL:" + ",".join(failed), valid=ok, correct=ok, failed=failed)
    return ev


# ---------- Data extraction ----------
SCHEMA_KEYS = {"order_id", "order_date", "delivery_sentiment", "packaging_sentiment", "issues"}

def parse_json_lenient(body):
    try:
        return json.loads(body)
    except Exception:
        m = re.search(r"\{.*\}", body or "", re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return None

def extraction_eval(expected, cot=False):
    def ev(resp):
        body = resp
        if cot:
            body = resp.split("FINAL JSON:")[-1].strip() if "FINAL JSON:" in resp else None
        strict = None
        if body is not None:
            try:
                strict = json.loads(body)
            except Exception:
                strict = None
        valid = isinstance(strict, dict) and set(strict) == SCHEMA_KEYS
        parsed = strict if isinstance(strict, dict) else parse_json_lenient(body)
        correct = False
        if isinstance(parsed, dict):
            correct = (
                (None if parsed.get("order_id") is None else str(parsed.get("order_id"))) == expected["order_id"]
                and (parsed.get("order_date") or None) == expected["order_date"]
                and parsed.get("delivery_sentiment") == expected["delivery_sentiment"]
                and parsed.get("packaging_sentiment") == expected["packaging_sentiment"]
                and isinstance(parsed.get("issues"), list)
                and set(parsed["issues"]) == set(expected["issues"])
            )
        if valid:
            canon = dict(strict); canon["issues"] = sorted(canon["issues"]) if isinstance(canon["issues"], list) else canon["issues"]
            key = json.dumps(canon, sort_keys=True)
        else:
            key = "INVALID:" + normalize(resp)
        return dict(key=key, valid=valid, correct=correct)
    return ev


def run_experiment(task, version, n, prompt, evaluator, case="base", temperature=DEFAULT_TEMPERATURE,
                   mode="modal", max_tokens=600):
    responses = run_many(prompt, n, f"{task}_{version}_{case}", temperature, max_tokens)
    evals = [evaluator(r) for r in responses]
    counts = Counter(e["key"] for e in evals)
    pass_rate = sum(e["valid"] for e in evals) / n
    row = dict(
        task=task, version=version, case=case, n=n, temperature=temperature,
        consistency=pass_rate if mode == "pass" else counts.most_common(1)[0][1] / n,
        format_valid=pass_rate,
        correct=sum(e["correct"] for e in evals) / n,
        unique_outputs=len(set(responses)),
    )
    RESULTS.append(row)
    RAW[(task, version, case, n)] = (responses, evals)
    return row


def table(rows, cols=("task", "version", "case", "n", "consistency", "format_valid", "correct", "unique_outputs")):
    df = pd.DataFrame(rows)[list(cols)]
    for c in ("consistency", "format_valid", "correct"):
        df[c] = (df[c] * 100).round(0).astype(int).astype(str) + "%"
    return df


def show_top(task, version, n, case="base", k=6, width=110):
    responses, _ = RAW[(task, version, case, n)]
    for text, c in Counter(responses).most_common(k):
        print(f"[{c:>2}x] {text[:width]!r}")

### Test inputs and ground truth
The three inputs from the lab brief are the **base** cases. Extra variation cases (Step 12) each carry a ground-truth label so I can measure correctness, not just agreement.

In [3]:
SENTIMENT_CASES = {
    "base":     ("I love this product! It's exactly what I needed.", "positive"),
    "negative": ("The package arrived two days late and the box was crushed. Not happy.", "negative"),
    "neutral":  ("I received my order today.", "neutral"),
    "mixed":    ("The screen is gorgeous but the battery barely lasts a day.", "neutral"),
    "sarcasm":  ("Oh great, another app update that deletes all my settings. Fantastic.", "negative"),
    "question": ("Does this jacket come in blue?", "neutral"),
}

PRODUCT_CASES = {
    "base":       dict(phrase="a wireless mouse that costs $29.99", name="Wireless mouse", price="$29.99",
                       info="Product: wireless mouse\nPrice: $29.99"),
    "bottle":     dict(phrase="a stainless steel water bottle (750 ml, keeps drinks cold for 24 hours) that costs $24.50",
                       name="Stainless steel water bottle", price="$24.50",
                       info="Product: stainless steel water bottle, 750 ml, keeps drinks cold for 24 hours\nPrice: $24.50"),
    "headphones": dict(phrase="noise-cancelling headphones (30-hour battery, Bluetooth 5.3) that cost $199.00",
                       name="Noise-cancelling headphones", price="$199.00",
                       info="Product: noise-cancelling headphones, 30-hour battery, Bluetooth 5.3\nPrice: $199.00"),
}

EXTRACTION_CASES = {
    "base": ("I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.",
             dict(order_id="12345", order_date="March 15", delivery_sentiment="positive",
                  packaging_sentiment="negative", issues=["damaged_packaging"])),
    "late_and_wet": ("My order #88421 from April 2nd arrived a week late and the box was soaked, but the headphones inside work fine.",
             dict(order_id="88421", order_date="April 2", delivery_sentiment="negative",
                  packaging_sentiment="negative", issues=["late_delivery", "damaged_packaging"])),
    "missing_fields": ("Got the wrong colour jacket. I placed the order on 3 January and nobody has answered my emails.",
             dict(order_id=None, order_date="January 3", delivery_sentiment=None,
                  packaging_sentiment=None, issues=["wrong_item"])),
}
print({k: len(v) for k, v in dict(sentiment=SENTIMENT_CASES, product=PRODUCT_CASES, extraction=EXTRACTION_CASES).items()}, "cases loaded")

{'sentiment': 6, 'product': 3, 'extraction': 3} cases loaded


### Step 2: Three zero-shot prompts (v1) — one call each

In [4]:
def sentiment_v1(msg):
    return f'Classify this customer message: "{msg}"'

def product_v1(c):
    return f"Create a product description for {c['phrase']}."

def extraction_v1(fb):
    return f'Extract information from this customer feedback: "{fb}"'

msg = SENTIMENT_CASES["base"][0]
fb = EXTRACTION_CASES["base"][0]
print("--- Sentiment v1 ---\n", call_openai(sentiment_v1(msg)))
print("\n--- Product v1 ---\n", call_openai(product_v1(PRODUCT_CASES["base"])))
print("\n--- Extraction v1 ---\n", call_openai(extraction_v1(fb)))

--- Sentiment v1 ---
 Customer Feedback: Positive Review



--- Product v1 ---
 **Product Name:** SwiftConnect Wireless Mouse

**Price:** $29.99

**Product Description:**

Elevate your computing experience with the SwiftConnect Wireless Mouse! Designed for both productivity enthusiasts and casual users, this sleek, ergonomic mouse combines style and functionality at an unbeatable price.

**Features:**

- **Wireless Convenience:** Enjoy seamless connectivity with advanced 2.4GHz wireless technology. Say goodbye to tangled wires and enjoy a clutter-free workspace with a range of up to 33 feet!

- **Ergonomic Design:** The SwiftConnect’s contoured shape provides comfort during extended use, reducing wrist strain and improving overall productivity. Its lightweight design makes it portable, perfect for on-the-go professionals.

- **Precision Tracking:** Equipped with a high-precision optical sensor, the SwiftConnect ensures smooth and accurate tracking on various surface types. Adjustable DPI settings allow you to customize sensitivity for tasks ra


--- Extraction v1 ---
 From the customer feedback, the following information can be extracted:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged


**Observation (single run).** Each v1 prompt "works" once, which is exactly why single-shot testing misleads. The sentiment answer
was *"classified as 'Positive Feedback' or 'Positive Review'"* (not one of the three labels), the product copy was ~300 words of markdown with an
invented brand name, and the extraction came back as a bulleted list rather than data a program can parse.

## Part 2 — Diagnosing failures by running the prompts repeatedly

### Step 3: 5 runs each

In [5]:
msg, exp_sent = SENTIMENT_CASES["base"]
prod = PRODUCT_CASES["base"]
fb, exp_ext = EXTRACTION_CASES["base"]

def run_v1(n):
    return [
        run_experiment("sentiment", "v1", n, sentiment_v1(msg), sentiment_eval(exp_sent)),
        run_experiment("product", "v1", n, product_v1(prod), product_eval(prod["price"]), mode="pass"),
        run_experiment("extraction", "v1", n, extraction_v1(fb), extraction_eval(exp_ext)),
    ]

rows5 = run_v1(5)
table(rows5)

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v1,base,5,20%,0%,100%,5
1,product,v1,base,5,0%,0%,0%,5
2,extraction,v1,base,5,20%,0%,0%,5


In [6]:
print("SENTIMENT v1 (5 runs) - distinct outputs:"); show_top("sentiment", "v1", 5)
print("\nPRODUCT v1 (5 runs) - first 2 outputs:")
for r in RAW[("product", "v1", "base", 5)][0][:2]: print("  >", r[:230].replace("\n", " | "), "...")
print("\nEXTRACTION v1 (5 runs) - first 2 outputs:")
for r in RAW[("extraction", "v1", "base", 5)][0][:2]: print("  >", r[:230].replace("\n", " | "))

SENTIMENT v1 (5 runs) - distinct outputs:
[ 1x] 'The customer message can be classified as **Positive Feedback**.'
[ 1x] 'This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."'
[ 1x] 'The customer message can be classified as **positive feedback** or **customer satisfaction**.'
[ 1x] 'The customer message can be classified as **positive feedback** or **satisfaction expression**.'
[ 1x] 'The customer message can be classified as "Positive Feedback" or "Satisfaction."'

PRODUCT v1 (5 runs) - first 2 outputs:
  > **Product Name: SwiftConnect Wireless Mouse** |  | **Price: $29.99** |  | **Product Description:** |  | Elevate your productivity and style with the SwiftConnect Wireless Mouse—a perfect blend of ergonomic design and cutting-edge wireless tec ...
  > **Product Name: SwiftClick Wireless Mouse** |  | **Price: $29.99** |  | **Description:** |  | Elevate your computing experience with the SwiftClick Wireless Mouse, meticulously designed for seamle

**5-run findings.**
- *Format:* 0/5 sentiment answers are a bare label; 0/5 extractions are JSON; every product description is a different length of bold-heavy markdown.
- *Content:* the sentiment direction is right every time (100% `correct`) but the wording is different every time (5 distinct outputs in 5 runs -> 20% consistency).
- *Variation types:* wording ("Positive Feedback" vs "positive feedback or customer satisfaction"), hedging with two labels, invented product names, and different key names in extraction ("Delivery Speed" vs "Issue").

### Step 4: 10 runs each

In [7]:
rows10 = run_v1(10)
table(rows5 + rows10)

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v1,base,5,20%,0%,100%,5
1,product,v1,base,5,0%,0%,0%,5
2,extraction,v1,base,5,20%,0%,0%,5
3,sentiment,v1,base,10,60%,0%,100%,7
4,product,v1,base,10,0%,0%,0%,10
5,extraction,v1,base,10,10%,0%,0%,10


In [8]:
print("SENTIMENT v1 (10 runs):"); show_top("sentiment", "v1", 10)
words = [len(r.split()) for r in RAW[("product", "v1", "base", 10)][0]]
print(f"\nPRODUCT v1 (10 runs): word counts min={min(words)} max={max(words)} mean={sum(words)/len(words):.0f}")
fmt = Counter("bullets/markdown" if ("**" in r or "- " in r) else "prose" for r in RAW[("extraction", "v1", "base", 10)][0])
print("EXTRACTION v1 (10 runs) output styles:", dict(fmt))

SENTIMENT v1 (10 runs):
[ 2x] 'This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."'
[ 2x] 'This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.'
[ 2x] 'This customer message can be classified as **positive feedback** or **customer satisfaction**.'
[ 1x] 'The customer message can be classified as **Positive Feedback**.'
[ 1x] "The customer's message can be classified as **Positive Feedback** or **Customer Satisfaction**."
[ 1x] 'The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.'

PRODUCT v1 (10 runs): word counts min=267 max=340 mean=298
EXTRACTION v1 (10 runs) output styles: {'bullets/markdown': 10}


**5 → 10 runs.** More runs surface more distinct outputs (sentiment: 5 → 7 unique; extraction: 5 → 10 unique). Note the sentiment "consistency" *rose* to 60% in this batch, which is
sampling luck, not improvement: the format validity is still 0%. That is the lesson of repeated sampling - a small batch can flatter a bad prompt. Product descriptions: 267-340 words, 0/10 pass the spec.

### Step 5: 15 runs each + failure analysis

In [9]:
rows15 = run_v1(15)
baseline = table(rows5 + rows10 + rows15)
baseline

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v1,base,5,20%,0%,100%,5
1,product,v1,base,5,0%,0%,0%,5
2,extraction,v1,base,5,20%,0%,0%,5
3,sentiment,v1,base,10,60%,0%,100%,7
4,product,v1,base,10,0%,0%,0%,10
5,extraction,v1,base,10,10%,0%,0%,10
6,sentiment,v1,base,15,27%,0%,100%,10
7,product,v1,base,15,0%,0%,0%,15
8,extraction,v1,base,15,20%,0%,0%,13


In [10]:
# ---- Failure analysis: what specifically goes wrong in the 15-run batch? ----
print("=== SENTIMENT v1: distinct outputs over 15 runs ===")
show_top("sentiment", "v1", 15, k=10)

print("\n=== PRODUCT v1: which spec checks fail (15 runs)? ===")
resp, evals = RAW[("product", "v1", "base", 15)]
fails = Counter(f for e in evals for f in e["failed"])
print(dict(fails))
words = [len(r.split()) for r in resp]
print(f"word count: min={min(words)}, max={max(words)}; distinct opening lines: {len(set(r.splitlines()[0] for r in resp))}/15")
print("openings:", [r.splitlines()[0][:60] for r in resp[:4]])

print("\n=== EXTRACTION v1: structure over 15 runs ===")
resp, evals = RAW[("extraction", "v1", "base", 15)]
print("strict-valid JSON:", sum(e["valid"] for e in evals), "/ 15")
print("mentions order number / date / delivery / packaging:",
      [sum(k in r.lower() for r in resp) for k in ("12345", "march 15", "fast", "damaged")])
print("distinct output styles:", Counter(("bullets" if re.search(r"^\s*[-*\d]", r, re.M) else "prose") + (" + markdown bold" if "**" in r else "") for r in resp))
print("sample:\n", resp[0][:300])

=== SENTIMENT v1: distinct outputs over 15 runs ===
[ 3x] 'The customer message can be classified as **Positive Feedback**.'
[ 2x] 'The customer message can be classified as "Positive Feedback" or "Satisfaction."'
[ 2x] 'This customer message can be classified as **positive feedback**.'
[ 2x] 'The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.'
[ 1x] 'This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."'
[ 1x] 'This customer message can be classified as **positive feedback** or **customer satisfaction**.'
[ 1x] 'The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."'
[ 1x] 'The customer message can be classified as positive feedback or a positive review.'
[ 1x] 'The customer message can be classified as **positive feedback** or **customer satisfaction**.'
[ 1x] 'The customer message can be classified as a **Positive Feedback** or **Customer Satisfaction**.'

=== PRODUCT v

### Failure analysis — v1 prompts, 15 runs (base inputs)

| Task | Consistency (5 / 10 / 15 runs) | Format valid | Correct | Unique outputs (15) | Specific failure patterns |
|---|---|---|---|---|---|
| Sentiment | 20% / 60% / 27% | 0% | 100% (direction), but see below | 10 | Never answers with a bare label. Always a sentence ("can be classified as **Positive Feedback**"), frequently hedged with a *second* category ("... or **Customer Satisfaction**"), varying capitalisation and markdown. |
| Product description | 0% / 0% / 0% | 0% | 0% | 15 | 220-347 words every time (spec: 40-100). Markdown bold and headings in all runs, 5-6 bullets rather than 3, exclamation marks and calls to action ("Order yours today!"), invented brand names (SwiftConnect, SwiftClick, SmoothScroll), invented specs (33 ft range, DPI settings), 11 different opening lines. |
| Data extraction | 20% / 10% / 20% | 0% | 0% (strict format) | 13 | 0/15 JSON. All 15 contain the right facts, but as markdown bullets (12 with bold), differing labels ("Packaging Condition" vs "Issue"), preamble sentences, and un-normalised values (`#12345`, `March 15th`) that downstream code would have to clean. |

**Reading the table.** Sentiment `correct` is 100% on the easy base message, but on the harder variations in Step 12 v1 stops using the sentiment labels at all
(it answers "complaint", "Order Update", "mixed feedback"), so the base message hides the real failure. The product 0% is partly by construction: the v1 prompt never asked for the format the spec checks - which is the point;
an unspecified prompt gives the model freedom, and it spends that freedom differently on every call.

## Part 3 — Iteration 1 (v2): clearer instructions, explicit format, constraints

### Steps 6-8: rewrite all three prompts

In [11]:
def sentiment_v2(msg):
    return f"""You are a sentiment classifier for customer-service messages.
Classify the sentiment of the customer message below as exactly one of: positive, negative, neutral.
Respond with ONLY that one lowercase word - no punctuation, no explanation.

Customer message: "{msg}"
"""

def product_v2(c):
    return f"""Write a product description for the item below, for an online store.

{c['info']}

Requirements:
- Start with a short headline of at most 8 words on its own line.
- Then a description of exactly 2 sentences.
- Then exactly 3 feature bullets, each starting with "- ", at most 8 words each.
- End with a line stating the price exactly as given.
- 50-80 words in total. Confident, plain tone. No exclamation marks, emojis or markdown formatting (no bold, no headings).
"""

def extraction_v2(fb):
    return f"""Extract structured data from the customer feedback below.

Return a JSON object with exactly these keys:
- "order_id": string, the order/item number without '#', or null if absent
- "order_date": string in the form "Month D" (e.g. "March 15"), or null
- "delivery_sentiment": "positive", "negative", "neutral" or null if delivery is not mentioned
- "packaging_sentiment": "positive", "negative", "neutral" or null if packaging is not mentioned
- "issues": array of zero or more of: "damaged_packaging", "late_delivery", "wrong_item", "missing_item", "damaged_product", "other"

Return only the JSON object.

Feedback: "{fb}"
"""

rows_v2 = [
    run_experiment("sentiment", "v2", 15, sentiment_v2(msg), sentiment_eval(exp_sent)),
    run_experiment("product", "v2", 15, product_v2(prod), product_eval(prod["price"]), mode="pass"),
    run_experiment("extraction", "v2", 15, extraction_v2(fb), extraction_eval(exp_ext)),
]
table(rows15 + rows_v2)

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v1,base,15,27%,0%,100%,10
1,product,v1,base,15,0%,0%,0%,15
2,extraction,v1,base,15,20%,0%,0%,13
3,sentiment,v2,base,15,100%,100%,100%,1
4,product,v2,base,15,100%,100%,100%,15
5,extraction,v2,base,15,80%,80%,100%,5


In [12]:
print("SENTIMENT v2:"); show_top("sentiment", "v2", 15)
resp, evals = RAW[("product", "v2", "base", 15)]
print("\nPRODUCT v2 failed checks:", dict(Counter(f for e in evals for f in e["failed"])))
print(resp[0])
resp, evals = RAW[("extraction", "v2", "base", 15)]
print("\nEXTRACTION v2: strict-valid", sum(e["valid"] for e in evals), "/15, correct", sum(e["correct"] for e in evals), "/15")
print("starts with code fence:", sum(r.startswith("```") for r in resp), "/15")
for text, c in Counter(e["key"] for e in evals).most_common(4): print(f"[{c:>2}x]", text[:160])

SENTIMENT v2:
[15x] 'positive'

PRODUCT v2 failed checks: {}
Ergonomic Wireless Mouse for Comfort and Precision

Experience seamless navigation with our Wireless Mouse, designed for all-day comfort and efficiency. Its sleek design and reliable connectivity make it an ideal choice for both work and leisure.

- Smooth, precise tracking  
- Easy plug-and-play setup  
- Long-lasting battery life  

Price: $29.99

EXTRACTION v2: strict-valid 12 /15, correct 15 /15
starts with code fence: 3 /15
[12x] {"delivery_sentiment": "positive", "issues": ["damaged_packaging"], "order_date": "March 15", "order_id": "12345", "packaging_sentiment": "negative"}
[ 2x] INVALID:json  orderid 12345  orderdate march 15  deliverysentiment positive  packagingsentiment negative  issues damagedpackaging
[ 1x] INVALID:json  orderid 12345  orderdate march 15  deliverysentiment positive  packagingsentiment negative  issues     damagedpackaging


### Iteration 1 (v2) — what changed and what happened
| Task | v1 → v2 consistency (15 runs) | What changed |
|---|---|---|
| Sentiment | 27% → **100%** | role + closed label set + "ONLY that one lowercase word, no punctuation" |
| Product | 0% → **100%** | explicit structure (headline, 2 sentences, exactly 3 bullets, price line), 50-80 words, tone and "no markdown/emoji/!" rules |
| Extraction | 20% → **80%** (12/15 valid) | explicit schema (keys, types, enums, null rules), "return only the JSON object" |

**Remaining v2 failure — extraction:** 3 of 15 responses wrapped the JSON in a ```` ```json ```` fence, so a plain `json.loads` raises. The *content* was right in all 15 (`correct` = 100%); only the wrapper broke parsing.
(The fresh 15-run batch in Step 13 showed 73% — the failure rate is real but noisy, which is why 15+ runs matter.) Fixes: `response_format={"type": "json_object"}` or a delimiter contract like v3's `FINAL JSON:`.
Plain instruction rewriting was enough for sentiment and product; extraction needed more.

## Part 4 — Iteration 2 (v3): few-shot examples, Chain-of-Thought and templates

### Step 9: Few-shot sentiment (5 examples incl. neutral, mixed and sarcasm)

In [13]:
def sentiment_v3(msg):
    return f"""You are a sentiment classifier for customer-service messages.
Classify each message as exactly one of: positive, negative, neutral.

Rules:
- positive: the customer expresses satisfaction or thanks.
- negative: the customer expresses dissatisfaction, a complaint or frustration. Sarcasm that is really a complaint is negative.
- neutral: no emotion (questions, factual statements) OR clearly mixed praise and complaint with no dominant side.
Respond with ONLY the one lowercase label - no punctuation, no explanation.

Message: "Terrible service, I waited 40 minutes and nobody helped me."
Sentiment: negative

Message: "Thanks, the replacement arrived and it works perfectly."
Sentiment: positive

Message: "I'd like to know the store opening hours on Sunday."
Sentiment: neutral

Message: "The camera is stunning but the battery dies by lunchtime."
Sentiment: neutral

Message: "Wow, three weeks to ship a pencil. Truly impressive."
Sentiment: negative

Message: "{msg}"
Sentiment:"""

r_s3 = run_experiment("sentiment", "v3", 15, sentiment_v3(msg), sentiment_eval(exp_sent))
table([r_s3])

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v3,base,15,100%,100%,100%,1


### Step 10: Chain-of-Thought extraction (reason first, then emit the JSON)

In [14]:
def extraction_v3(fb):
    return f"""You extract structured data from customer feedback. Work through it step by step, then output JSON.

Schema (exactly these keys):
- "order_id": string without '#', or null if no order/item number is stated
- "order_date": "Month D" (e.g. "March 15"; convert "3 January" -> "January 3", drop ordinal suffixes), or null
- "delivery_sentiment": "positive" | "negative" | "neutral" | null (null if delivery is not mentioned)
- "packaging_sentiment": "positive" | "negative" | "neutral" | null (null if packaging is not mentioned)
- "issues": array drawn only from: "damaged_packaging", "late_delivery", "wrong_item", "missing_item", "damaged_product", "other"

Procedure - write it out under a line "REASONING:":
1. Quote any order/item number, or write "none".
2. Quote any date and rewrite it as "Month D", or write "none".
3. Say what is stated about delivery and whether that is positive, negative or neutral (or "not mentioned").
4. Say what is stated about packaging, the same way.
5. List which allowed issue labels apply.
Then write a line "FINAL JSON:" followed by the raw JSON object only (no code fences, nothing after it).

Example
Feedback: "Item #777 bought on 9th June never showed up, but the seller was polite."
REASONING:
1. Order number: 777
2. Date: "9th June" -> "June 9"
3. Delivery: the item never showed up -> negative
4. Packaging: not mentioned
5. Issues: missing_item
FINAL JSON:
{{"order_id": "777", "order_date": "June 9", "delivery_sentiment": "negative", "packaging_sentiment": null, "issues": ["missing_item"]}}

Feedback: "{fb}"
REASONING:"""

r_e3 = run_experiment("extraction", "v3", 15, extraction_v3(fb), extraction_eval(exp_ext, cot=True), max_tokens=700)
table([r_e3])

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,extraction,v3,base,15,100%,100%,100%,2


In [15]:
print(RAW[("extraction", "v3", "base", 15)][0][0])

1. Order number: 12345
2. Date: "March 15th" -> "March 15"
3. Delivery: delivery was fast -> positive
4. Packaging: the packaging was damaged -> negative
5. Issues: damaged_packaging
FINAL JSON:
{"order_id": "12345", "order_date": "March 15", "delivery_sentiment": "positive", "packaging_sentiment": "negative", "issues": ["damaged_packaging"]}


### Step 11: Few-shot + template for the product description

In [16]:
def product_v3(c):
    return f"""You write product descriptions for an online store. Follow the template exactly.

Template:
<Headline, max 8 words>
<Exactly 2 sentences, plain confident tone, no exclamation marks, no emojis, no markdown>
- <feature, max 8 words>
- <feature, max 8 words>
- <feature, max 8 words>
Price: <price exactly as given>

Example 1
Product: mechanical keyboard, hot-swappable switches
Price: $89.00
Output:
Typing that feels as good as it sounds
Built around hot-swappable switches, this keyboard lets you tune every key to your taste. Its sturdy aluminium frame keeps each keystroke steady through long work days.
- Hot-swappable mechanical switches
- Sturdy aluminium frame
- Comfortable full-size layout
Price: $89.00

Example 2
Product: USB-C hub with 7 ports
Price: $45.50
Output:
One cable, seven connections
This compact hub turns a single USB-C port into everything your desk needs. Plug in displays, drives and chargers without hunting for adapters.
- Seven ports in one slim body
- 4K HDMI video output
- 100W pass-through charging
Price: $45.50

Now write the description.
{c['info']}
Output:
"""

r_p3 = run_experiment("product", "v3", 15, product_v3(prod), product_eval(prod["price"]), mode="pass")
print(RAW[("product", "v3", "base", 15)][0][0])
table([r_p3])

Precision and comfort in your hand  
This wireless mouse offers seamless navigation and ergonomic design for all-day use. Enjoy a smooth experience without the hassle of tangled wires.  
- Ergonomic design for comfort  
- Wireless connectivity for convenience  
- Long-lasting battery life  
Price: $29.99


,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,product,v3,base,15,100%,100%,100%,15


### Iteration 2 (v3) — results and which technique earned its keep
| Task | v1 | v2 | v3 | Technique |
|---|---|---|---|---|
| Sentiment | 27% | 100% | **100%** (1 unique output) | few-shot (5 examples incl. neutral, mixed, sarcasm) + explicit rules |
| Extraction | 20% | 80% | **100%** (identical JSON every run) | Chain-of-Thought: quote → normalise → label → emit JSON after a `FINAL JSON:` marker, one worked example |
| Product | 0% | 100% | **100%** | few-shot (2 examples) + exact template |

**Honest reading.** On the easy *base* inputs v2 was already ~100% for sentiment and product, so v3's gain there is invisible; v3's value appears in the variations (Step 12) and in extraction, where
CoT removed the code-fence failure (the reasoning is kept apart from the machine-readable part) and made normalisation explicit (`March 15th` → `March 15`).
The 2 "unique" raw extraction outputs differ only in the reasoning text — the parsed JSON was identical in 15/15.

In [17]:
v123 = table([r for r in RESULTS if r["case"] == "base" and r["n"] == 15])
v123.sort_values(["task", "version"]).reset_index(drop=True)

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,extraction,v1,base,15,20%,0%,0%,13
1,extraction,v2,base,15,80%,80%,100%,5
2,extraction,v3,base,15,100%,100%,100%,2
3,product,v1,base,15,0%,0%,0%,15
4,product,v2,base,15,100%,100%,100%,15
5,product,v3,base,15,100%,100%,100%,15
6,sentiment,v1,base,15,27%,0%,100%,10
7,sentiment,v2,base,15,100%,100%,100%,1
8,sentiment,v3,base,15,100%,100%,100%,1


## Part 5 — Tuning for variations

### Step 12: Task variations (10 runs each, v1 vs v3, with ground truth)

In [18]:
VAR_N = 10
var_rows = []
for case, (m, label) in SENTIMENT_CASES.items():
    if case == "base": continue
    var_rows.append(run_experiment("sentiment", "v1", VAR_N, sentiment_v1(m), sentiment_eval(label), case))
    var_rows.append(run_experiment("sentiment", "v3", VAR_N, sentiment_v3(m), sentiment_eval(label), case))
for case, c in PRODUCT_CASES.items():
    if case == "base": continue
    var_rows.append(run_experiment("product", "v1", VAR_N, product_v1(c), product_eval(c["price"]), case, mode="pass"))
    var_rows.append(run_experiment("product", "v3", VAR_N, product_v3(c), product_eval(c["price"]), case, mode="pass"))
for case, (f, exp) in EXTRACTION_CASES.items():
    if case == "base": continue
    var_rows.append(run_experiment("extraction", "v1", VAR_N, extraction_v1(f), extraction_eval(exp), case))
    var_rows.append(run_experiment("extraction", "v3", VAR_N, extraction_v3(f), extraction_eval(exp, cot=True), case, max_tokens=700))
variations = table(var_rows)
variations

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,sentiment,v1,negative,10,50%,0%,0%,7
1,sentiment,v3,negative,10,100%,100%,100%,1
2,sentiment,v1,neutral,10,10%,0%,0%,10
3,sentiment,v3,neutral,10,100%,100%,100%,1
4,sentiment,v1,mixed,10,10%,0%,0%,10
5,sentiment,v3,mixed,10,100%,100%,100%,1
6,sentiment,v1,sarcasm,10,10%,0%,70%,10
7,sentiment,v3,sarcasm,10,100%,100%,100%,1
8,sentiment,v1,question,10,30%,0%,0%,8
9,sentiment,v3,question,10,100%,100%,100%,1


In [19]:
# Where did v3 still fall short on variations?
for r in var_rows:
    if r["version"] == "v3" and (r["consistency"] < 1 or r["correct"] < 1):
        print(f"{r['task']}/{r['case']}: consistency={r['consistency']:.0%}, correct={r['correct']:.0%}")
        if r["task"] == "sentiment":
            show_top("sentiment", "v3", VAR_N, r["case"], k=3)
        elif r["task"] == "extraction":
            resp, ev = RAW[("extraction", "v3", r["case"], VAR_N)]
            for t, c in Counter(e["key"] for e in ev).most_common(3): print(f"  [{c}x] {t[:170]}")
        else:
            resp, ev = RAW[("product", "v3", r["case"], VAR_N)]
            print("  failed checks:", dict(Counter(f for e in ev for f in e["failed"])))

extraction/missing_fields: consistency=50%, correct=50%
  [5x] {"delivery_sentiment": null, "issues": ["wrong_item"], "order_date": "January 3", "order_id": null, "packaging_sentiment": null}
  [3x] {"delivery_sentiment": "negative", "issues": ["wrong_item"], "order_date": "January 3", "order_id": null, "packaging_sentiment": null}
  [2x] {"delivery_sentiment": "neutral", "issues": ["wrong_item"], "order_date": "January 3", "order_id": null, "packaging_sentiment": null}


### What the variations show
- **Sentiment v3 is robust (100% consistent and correct on all 5 new messages)**, including sarcasm and a mixed review. v1 was 0% correct on four of five because it stopped using sentiment labels altogether ("complaint", "Order Update", "mixed feedback"); on sarcasm it was 70% correct but 10% consistent.
  Caveat: "mixed → neutral" is a *policy* I encoded in the prompt and one few-shot example; a client who wants a fourth `mixed` label would change the prompt, not the model.
- **Product v3 holds up on new products** (100% spec pass on the bottle and headphones; every output is different text).
- **Extraction v3 still has a gap:** on `missing_fields` only 50% of runs were correct. The customer never mentioned delivery, but 5 of 10 runs invented a `delivery_sentiment` ("negative" ×3, "neutral" ×2) — the model projected "nobody answered my emails" onto delivery. Date normalisation (`3 January` → `January 3`) and `null` order id were handled correctly. This is a *specification* gap, not randomness.

### Step 12b: Tuning v3 → v3.1 where the variations exposed weaknesses
Two gaps showed up: (1) extraction invents a `delivery_sentiment` when the customer complains about *customer service* (`missing_fields`),
and (2) product outputs cluster just above the 40-word floor because the examples are short. Fix each with a targeted change,
then **re-test every case** (15 runs) to check for regressions.

In [20]:
extraction_v3b_rule = """
Scope rules (important):
- "delivery_sentiment" is ONLY about shipping/delivery (speed, lateness, whether it arrived). "packaging_sentiment" is ONLY about the box/packaging.
- Complaints about customer service, unanswered emails, price or the product itself are NOT about delivery or packaging -> use null for those fields
  (list the problem under "issues" if a label fits, e.g. wrong_item).
"""

def extraction_v3b(fb):
    p = extraction_v3(fb)
    p = p.replace("Procedure - write it out", extraction_v3b_rule + "\nProcedure - write it out")
    p = p.replace('3. Say what is stated about delivery and whether that is positive, negative or neutral (or "not mentioned").',
                  '3. Say what is stated about shipping/delivery itself and whether that is positive, negative or neutral (or "not mentioned" - service complaints do not count).')
    return p


def product_v3b(c):
    p = product_v3(c)
    # longer, mid-range examples (~60 words) + explicit length target
    p = p.replace("<Exactly 2 sentences, plain confident tone, no exclamation marks, no emojis, no markdown>",
                  "<Exactly 2 sentences of 18-25 words each, plain confident tone, no exclamation marks, no emojis, no markdown>")
    p = p.replace("Template:", "Target length: 55-75 words in total.\n\nTemplate:")
    p = p.replace("Built around hot-swappable switches, this keyboard lets you tune every key to your taste. Its sturdy aluminium frame keeps each keystroke steady through long work days.",
                  "Built around hot-swappable switches, this keyboard lets you tune the feel of every single key to match your own taste. Its sturdy aluminium frame keeps each keystroke steady and quiet through long, demanding work days.")
    p = p.replace("This compact hub turns a single USB-C port into everything your desk needs. Plug in displays, drives and chargers without hunting for adapters.",
                  "This compact hub turns a single USB-C port into everything your desk could possibly need for the day ahead. Plug in displays, drives and chargers at the same time without ever hunting for adapters.")
    return p

import statistics
ex_words = [len(m.split()) for m in re.findall(r"Example \d\n.*?Output:\n(.*?)\n\nExample|Output:\n(.*?)\n\nNow", product_v3b(PRODUCT_CASES["base"]), re.S) for m in m if m]
print("v3.1 product example lengths (words):", ex_words)

tune_rows = []
for case, (f, exp) in EXTRACTION_CASES.items():
    tune_rows.append(run_experiment("extraction", "v3.1", 15, extraction_v3b(f), extraction_eval(exp, cot=True), f"tune_{case}", max_tokens=700))
for case, c in PRODUCT_CASES.items():
    tune_rows.append(run_experiment("product", "v3.1", 15, product_v3b(c), product_eval(c["price"]), f"tune_{case}", mode="pass"))
tuned = table(tune_rows)
tuned

v3.1 product example lengths (words): [57, 56]


,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,extraction,v3.1,tune_base,15,100%,100%,100%,1
1,extraction,v3.1,tune_late_and_wet,15,100%,100%,100%,5
2,extraction,v3.1,tune_missing_fields,15,100%,100%,100%,3
3,product,v3.1,tune_base,15,100%,100%,100%,15
4,product,v3.1,tune_bottle,15,100%,100%,100%,15
5,product,v3.1,tune_headphones,15,100%,100%,100%,15


In [21]:
# v3 vs v3.1 side by side (v3 numbers come from the earlier runs: base 15 runs, variations 10 runs)
v3_prev = {(r["task"], r["case"]): r for r in RESULTS if r["version"] == "v3" and r["case"] in
           ("base", "late_and_wet", "missing_fields", "bottle", "headphones") and r["temperature"] == 1.0 and r["n"] in (10, 15)}
cmp_rows = []
for r in tune_rows:
    case = r["case"].replace("tune_", "")
    old = v3_prev[(r["task"], case)]
    cmp_rows.append(dict(task=r["task"], case=case,
                         v3_consistency=f"{old['consistency']:.0%}", v31_consistency=f"{r['consistency']:.0%}",
                         v3_correct=f"{old['correct']:.0%}", v31_correct=f"{r['correct']:.0%}"))
pd.DataFrame(cmp_rows)

,task,case,v3_consistency,v31_consistency,v3_correct,v31_correct
0,extraction,base,100%,100%,100%,100%
1,extraction,late_and_wet,100%,100%,100%,100%
2,extraction,missing_fields,50%,100%,50%,100%
3,product,base,100%,100%,100%,100%
4,product,bottle,100%,100%,100%,100%
5,product,headphones,100%,100%,100%,100%


**Result:** the scope rule ("delivery/packaging sentiment only for shipping/box; service complaints → null") took `missing_fields` from 50% to **100%** with no regression on the other extraction cases,
and the longer examples + explicit "55-75 words" target moved product outputs into the middle of the allowed range (48-65 words) with 100% spec-pass on all three products.
**Caveat:** I tuned v3.1 using the same failing case I diagnosed it on, so `missing_fields` is in-sample. To trust it I would add several more "field not mentioned" cases.

In [22]:
resp, ev = RAW[("extraction", "v3.1", "tune_missing_fields", 15)]
for t, c in Counter(e["key"] for e in ev).most_common(3): print(f"[{c}x] {t[:170]}")
for task, case in [("product", "tune_base"), ("product", "tune_bottle")]:
    resp, ev = RAW[(task, "v3.1", case, 15)]
    w = [len(r.split()) for r in resp]
    print(f"{case}: words min={min(w)} max={max(w)}; failed checks: {dict(Counter(f for e in ev for f in e['failed']))}")

[15x] {"delivery_sentiment": null, "issues": ["wrong_item"], "order_date": "January 3", "order_id": null, "packaging_sentiment": null}
tune_base: words min=48 max=59; failed checks: {}
tune_bottle: words min=52 max=65; failed checks: {}


### Step 13: Final evaluation — fresh 15-run batches of v1, v2, v3 on the base inputs

In [23]:
# Fresh independent batches (new labels -> new samples) so the final numbers are not the same draws used while iterating.
final = []
for version, sb, pb, eb, cot in [("v1", sentiment_v1, product_v1, extraction_v1, False),
                                 ("v2", sentiment_v2, product_v2, extraction_v2, False),
                                 ("v3", sentiment_v3, product_v3, extraction_v3, True)]:
    final += [
        run_experiment("sentiment", version, 15, sb(msg), sentiment_eval(exp_sent), "final"),
        run_experiment("product", version, 15, pb(prod), product_eval(prod["price"]), "final", mode="pass"),
        run_experiment("extraction", version, 15, eb(fb), extraction_eval(exp_ext, cot=cot), "final", max_tokens=700),
    ]
final_df = pd.DataFrame(final)
display(table(final).sort_values(["task", "version"]).reset_index(drop=True))

pivot = final_df.pivot(index="task", columns="version", values="consistency")
pivot["improvement v1->v3 (pts)"] = ((pivot["v3"] - pivot["v1"]) * 100).round(0).astype(int)
display((pivot[["v1", "v2", "v3"]] * 100).round(0).astype(int).assign(**{"v1->v3 (pts)": pivot["improvement v1->v3 (pts)"]}))

print("\nSuccess criterion: v3 consistency > 80% on every task")
for _, r in final_df[final_df.version == "v3"].iterrows():
    print(f"  {r['task']:<11} {r['consistency']:.0%}  {'PASS' if r['consistency'] > 0.8 else 'FAIL'}")

,task,version,case,n,consistency,format_valid,correct,unique_outputs
0,extraction,v1,final,15,33%,0%,0%,12
1,extraction,v2,final,15,73%,73%,100%,5
2,extraction,v3,final,15,100%,100%,100%,1
3,product,v1,final,15,0%,0%,0%,15
4,product,v2,final,15,100%,100%,100%,15
5,product,v3,final,15,93%,93%,93%,15
6,sentiment,v1,final,15,33%,0%,100%,12
7,sentiment,v2,final,15,100%,100%,100%,1
8,sentiment,v3,final,15,100%,100%,100%,1


version,v1,v2,v3,v1->v3 (pts)
task,,,,
extraction,33,73,100,67
product,0,100,93,93
sentiment,33,100,100,67



Success criterion: v3 consistency > 80% on every task
  sentiment   100%  PASS
  product     93%  PASS
  extraction  100%  PASS


In [24]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
tasks = ["sentiment", "product", "extraction"]
width = 0.26
for i, v in enumerate(["v1", "v2", "v3"]):
    vals = [final_df[(final_df.task == t) & (final_df.version == v)]["consistency"].iloc[0] * 100 for t in tasks]
    bars = ax.bar([x + (i - 1) * width for x in range(3)], vals, width, label=v)
    ax.bar_label(bars, fmt="%.0f", fontsize=8)
ax.axhline(80, ls="--", color="gray", lw=1); ax.text(2.45, 81, "80% target", ha="right", fontsize=8, color="gray")
ax.set_xticks(range(3)); ax.set_xticklabels(tasks); ax.set_ylabel("consistency (%)"); ax.set_ylim(0, 110)
ax.set_title("Consistency over 15 runs, by prompt version"); ax.legend(loc="upper left", ncol=3)
plt.tight_layout(); plt.savefig("results/consistency_comparison.png", dpi=150)

pd.DataFrame(RESULTS).to_csv("results/summary.csv", index=False)
json.dump({f"{k[0]}|{k[1]}|{k[2]}|{k[3]}": v[0] for k, v in RAW.items()}, open("results/raw_responses.json", "w"), indent=1)
print("saved results/summary.csv, results/raw_responses.json, results/consistency_comparison.png")

saved results/summary.csv, results/raw_responses.json, results/consistency_comparison.png


### Final comparison (fresh 15-run batches, base inputs)
| Task | v1 | v2 | v3 | Improvement v1 → v3 |
|---|---|---|---|---|
| Sentiment | 33% | 100% | **100%** | +67 pts |
| Product description | 0% | 100% | **93%** | +93 pts |
| Data extraction | 33% | 73% | **100%** | +67 pts |

All three v3 prompts exceed the 80% target. Two points to be straight about:
1. The v1 sentiment/extraction numbers differ from the earlier 27%/20% only because of sampling noise (v1 is never valid in either batch) - another reason to quote ranges, not single numbers.
2. **v3 product dipped to 93% in the fresh batch** (earlier batch: 100%). The one failure was a 38-word output, just under my 40-word floor; the examples were ~45 words so outputs hugged the lower bound. That is what v3.1 fixed (Step 12b).

## Bonus Challenge 1 — Temperature experiment (10 runs per cell, v1 vs v3, base inputs)

In [25]:
temp_rows = []
for t in (0, 0.3, 0.7, 1.0):
    for version, sb, pb, eb, cot in [("v1", sentiment_v1, product_v1, extraction_v1, False),
                                     ("v3", sentiment_v3, product_v3, extraction_v3, True)]:
        temp_rows += [
            run_experiment("sentiment", version, 10, sb(msg), sentiment_eval(exp_sent), f"temp{t}", temperature=t),
            run_experiment("product", version, 10, pb(prod), product_eval(prod["price"]), f"temp{t}", temperature=t, mode="pass"),
            run_experiment("extraction", version, 10, eb(fb), extraction_eval(exp_ext, cot=cot), f"temp{t}", temperature=t, max_tokens=700),
        ]
tdf = pd.DataFrame(temp_rows)
(tdf.pivot_table(index=["task", "version"], columns="temperature", values="consistency") * 100).round(0).astype(int)

temperature         0.0  0.3  0.7  1.0
task       version                    
extraction v1        90  100   50   30
           v3       100  100  100  100
product    v1         0    0    0    0
           v3       100  100  100  100
sentiment  v1        50   50   60   40
           v3       100  100  100  100

In [26]:
pd.DataFrame(RESULTS).to_csv("results/summary.csv", index=False)
(tdf.pivot_table(index=["task", "version"], columns="temperature", values="unique_outputs")).astype(int).rename_axis(columns="unique outputs @ temperature")

unique outputs @ temperature  0.0  0.3  0.7  1.0
task       version                              
extraction v1                   2    2    7    8
           v3                   1    1    1    1
product    v1                   8   10   10   10
           v3                   1    7   10   10
sentiment  v1                   2    3    6    8
           v3                   1    1    1    1

### Temperature findings
- **Temperature does not fix an under-specified prompt.** v1 product stays at 0% spec-pass at every temperature (0 → 1.0), and v1 extraction at temperature 0.3 is 100% *consistent* but 0% valid JSON: perfectly repeatable, perfectly unusable. Consistency and correctness are different things, which is why the tables report both.
- v1 sentiment is only 40-60% consistent at every temperature, and 0% format-valid at all of them - lowering temperature never produced a bare label.
- **v3 holds 100% at every temperature tested** (0, 0.3, 0.7, 1.0) for all three tasks. Notice the second table: v3 product has 1 unique output at temperature 0 but 10 at temperature 1.0 - the *content* varies while the *format* does not, which is the behaviour you want for generation.
- Practical rule: fix the prompt first, then use temperature 0-0.3 for classification/extraction and a higher value only for generation where variety is wanted.